# RuBERT-tiny-turbo - AWS Marketplace

Deploys **RuBERT-tiny-turbo** from AWS Marketplace as a SageMaker endpoint inside **your own AWS account**. Your data stays in your account and there are no external API calls.

Russian text embeddings from RuBERT-tiny-turbo: 312-dimension vectors, 29.2M parameters, 116.8MB, deployable on ml.t2.medium. Measured Spearman 0.784 on the Russian STS Benchmark (word overlap 0.511); sample cosine 0.96 similar vs 0.64 unrelated. MIT, private in your VPC.

## Prerequisites

1. Subscribe to the product in AWS Marketplace.
2. Copy the **model package ARN** shown on the product's launch page for your Region.
3. Run this notebook with a role that has `AmazonSageMakerFullAccess`.

Cost: software fee $0.05/hr plus the SageMaker instance (ml.t2.medium real-time, ml.m5.large batch). Delete the endpoint when done.

In [ ]:
!pip install -qU sagemaker boto3

In [ ]:
import json

import boto3
import sagemaker
from sagemaker import ModelPackage

# Paste the model package ARN from the product's launch page for YOUR Region.
MODEL_PACKAGE_ARN = "<paste-model-package-arn-here>"

INSTANCE_TYPE = "ml.t2.medium"  # smallest real-time type, enough for this model
ENDPOINT_NAME = "rubert-tiny-turbo-embeddings"

session = sagemaker.Session()
role = sagemaker.get_execution_role()
print("region:", session.boto_region_name)

## 2. Deploy a real-time endpoint

Creating the endpoint takes a few minutes. It bills per hour while it exists, so do not skip section 5.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)
print("endpoint ready:", ENDPOINT_NAME)

## 3. Embed text

The endpoint accepts `{"inputs": ["text", ...], "normalize": true}` and returns `embeddings`, `dimension` and `normalized`. With `normalize` true, cosine similarity is a plain dot product.

In [ ]:
runtime = boto3.client("sagemaker-runtime")

PAYLOAD = {'inputs': ['Девушка расчесывает волосы.', 'Девушка укладывает волосы.'], 'normalize': True}

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(PAYLOAD),
)
result = json.loads(response["Body"].read())
print(json.dumps(result)[:600])

import numpy as np
E = np.array(result["embeddings"])
print("dimension:", result["dimension"])
print("cosine(0,1):", float(E[0] @ E[1]))

## 4. Batch transform for offline workloads

Batch transform avoids paying for an always-on endpoint (ml.m5.large). The input file is sent as ONE request, so put a single JSON request object (same shape as above) in the file.

In [ ]:
# transformer = model.transformer(
#     instance_count=1,
#     instance_type="ml.m5.large",
#     output_path=f"s3://{session.default_bucket()}/rubert-tiny-turbo-embeddings/",
#     strategy="SingleRecord",
# )
# transformer.transform(
#     data=f"s3://{session.default_bucket()}/rubert-tiny-turbo-embeddings-input/request.jsonl",
#     content_type="application/json",
# )
# transformer.wait()

## 5. Clean up

Delete the endpoint when you are done. It bills for as long as it is running.

In [ ]:
predictor.delete_model()
predictor.delete_endpoint()
print("deleted:", ENDPOINT_NAME)